# Liu2024 - S-JEPA x Riemannian hybrid (head-to-head with the Riemannian baseline)

Combines the two methods at the **feature level**, then classifies with a shrinkage-LDA so it stays
data-efficient (no SGD head -> no collapse):

- **S-JEPA branch:** the *frozen pretrained* local encoder (channel-agnostic, so no random spatial layer
  is involved) -> per-channel feature maps -> mean-pooled over time tokens -> embedding vector.
- **Riemannian branch:** filter-bank spatial covariances -> tangent-space vectors (concatenated over bands).
- **Fusion:** standardize each branch on the train fold, concatenate, shrinkage-LDA.

It reports **Riemannian-only**, **S-JEPA-only**, and **Fusion** on the *same* folds so you can see the
difference directly. Run the companion `liu2024_twfb_dgfmdm_reproduction.ipynb` for the full per-subject
TW+FB-selected FgMDM (the closest thing to the 72% method); this notebook uses a fixed filter bank +
tangent-LDA as the Riemannian branch so all three share one classifier and fusion is apples-to-apples.

> Requires pyRiemann: `pip install pyriemann`

# 1. Setup

In [ ]:
import os, re, sys, json, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch

from scipy.io import loadmat
import mne
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

from braindecode.models import SignalJEPA_PreLocal
try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
except Exception as e:
    raise ImportError("pyRiemann is required: pip install pyriemann") from e
print("torch", torch.__version__)


# 2. Configuration

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-sjepa-riemannian-hybrid"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "subjects_to_use": None, "exclude_subjects": [],

    # ---- signal / window (identical to the TWFB notebook for comparability) ----
    "source_unit": "microvolts", "reference_mode": "average",
    "resample_sfreq": 128, "mi_window_start_s": 2.0, "target_window_samples": 537,
    "demean_mode": "baseline_window_mean", "baseline_window_s": [0.0, 2.0],

    # ---- S-JEPA branch ----
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "pretrained_mode": "from_pretrained",       # from_pretrained, random (control)

    # ---- Riemannian branch ----
    "filter_bands": [[8,12],[12,16],[16,24],[24,30]],
    "cov_estimator": "oas",

    # ---- classifier / evaluation ----
    "lda_shrinkage": "auto", "cv_splits": 5, "seed": 2026,
}
LIU_SFREQ=500; SFREQ=float(CONFIG["resample_sfreq"]); WIN=int(CONFIG["target_window_samples"])
ART=Path(CONFIG["artifact_dir"])/datetime.now().strftime("%Y%m%d_%H%M"); ART.mkdir(parents=True,exist_ok=True)
with open(ART/"config.json","w") as f: json.dump(CONFIG,f,indent=2,default=str)
DEVICE=(torch.device("cuda") if torch.cuda.is_available()
        else torch.device("mps") if torch.backends.mps.is_available() and torch.backends.mps.is_built()
        else torch.device("cpu"))
print("device",DEVICE,"| bands",CONFIG["filter_bands"])


# 3. Data: loader + preprocessing (shared with the TWFB notebook)

In [ ]:
SOURCE_EEG_INDICES=[i for i in range(30) if i!=17]
NAMES30=["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4",
         "CPz","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
EEG_NAMES=[n for i,n in enumerate(NAMES30) if i!=17]; N_CH=len(EEG_NAMES)
def find_mats(root): root=Path(root); return sorted(root.rglob("*.mat")) if root.exists() else []
def sid_from_path(p):
    m=re.search(r"sub[-_ ]?(\d{1,2})",str(p),re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+",Path(p).stem)[-1])
def _walk(o,pre=""):
    if isinstance(o,dict):
        for k,v in o.items():
            if str(k).startswith("__"): continue
            n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif hasattr(o,"_fieldnames"):
        for k in o._fieldnames:
            v=getattr(o,k); n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif isinstance(o,np.ndarray):
        if o.dtype==object and o.size==1: yield from _walk(o.item(),pre)
        elif o.dtype==object:
            for idx,it in np.ndenumerate(o): yield from _walk(it,f"{pre}{idx}")
def load_subject(path):
    mat=loadmat(path,squeeze_me=True,struct_as_record=False)
    arrs=[(n,np.asarray(v)) for n,v in _walk(mat) if isinstance(v,np.ndarray) and v.dtype!=object]
    raws=[a for n,a in arrs if a.ndim==3]; labs=[a for n,a in arrs if a.ndim in (1,2) and np.asarray(a).size in (39,40)]
    if not raws or not labs: raise KeyError(path)
    raw=max(raws,key=lambda a:max(a.shape)); labels=np.asarray(labs[0]).astype(int).ravel()
    if raw.shape[0] not in (39,40):
        ax=[i for i,s in enumerate(raw.shape) if s in (39,40)]
        if ax: raw=np.moveaxis(raw,ax[0],0)
    if int(np.argmax(raw.shape[1:])+1)!=2: raw=np.moveaxis(raw,int(np.argmax(raw.shape[1:])+1),2)
    return raw.astype(np.float64), labels
def to_zero(l):
    u=set(np.unique(l).tolist())
    if u.issubset({1,2}): return l-1
    if u.issubset({0,1}): return l
    raise ValueError(u)
def preprocess(raw, labels):
    X=raw[:,SOURCE_EEG_INDICES,:].astype(np.float64); n=X.shape[0]
    b0,b1=CONFIG["baseline_window_s"]; s0,s1=int(b0*LIU_SFREQ),int(b1*LIU_SFREQ)
    if CONFIG["demean_mode"]=="baseline_window_mean": X=X-X[:,:,s0:s1].mean(-1,keepdims=True)
    cont=(X*1e-6).transpose(1,0,2).reshape(len(SOURCE_EEG_INDICES),-1)
    r=mne.io.RawArray(cont,mne.create_info(EEG_NAMES,LIU_SFREQ,["eeg"]*len(EEG_NAMES)),verbose=False)
    if CONFIG["reference_mode"]=="average": r.set_eeg_reference("average",projection=False,verbose=False)
    r.resample(SFREQ,verbose=False)
    d=r.get_data()*1e6; per=d.shape[1]//n; d=d[:,:n*per]
    Xrs=d.reshape(len(SOURCE_EEG_INDICES),n,per).transpose(1,0,2)
    st=int(round(CONFIG["mi_window_start_s"]*SFREQ)); sp=st+WIN
    return Xrs[:,:,st:sp].astype(np.float64), to_zero(labels).astype(int)
MATS=find_mats(CONFIG["source_extract_dir"])
if not MATS: raise FileNotFoundError(CONFIG["source_extract_dir"])
use=None if CONFIG["subjects_to_use"] is None else set(CONFIG["subjects_to_use"]); excl=set(CONFIG["exclude_subjects"])
SUBJECTS={}
for p in MATS:
    s=sid_from_path(p)
    if (use is not None and s not in use) or s in excl: continue
    raw,lab=load_subject(p); SUBJECTS[s]=preprocess(raw,lab)
ALL=sorted(SUBJECTS); print(f"loaded {len(ALL)} subjects | X={SUBJECTS[ALL[0]][0].shape}")


# 4. S-JEPA branch: frozen pretrained embeddings

We call the pretrained `feature_encoder` directly on the raw channels (it encodes each channel
independently, so no random spatial layer is involved), then mean-pool over the time tokens and flatten
to a per-trial embedding. This is purely the pretrained representation - no training, no collapse.

In [ ]:
_info=mne.create_info(EEG_NAMES,SFREQ,["eeg"]*N_CH)
_info.set_montage(mne.channels.make_standard_montage("standard_1020"),match_case=False,on_missing="ignore")
CHS_INFO=_info["chs"]
def build_encoder():
    kw=dict(n_chans=N_CH,chs_info=CHS_INFO,n_times=WIN,n_outputs=2)
    if CONFIG["pretrained_mode"]=="from_pretrained":
        m=SignalJEPA_PreLocal.from_pretrained(CONFIG["pretrained_repo_id"],**kw,strict=False)
    else:
        m=SignalJEPA_PreLocal(**kw)
    return m.to(DEVICE).eval()
ENCODER=build_encoder()
fe_params=sum(p.numel() for n,p in ENCODER.named_parameters() if n.startswith("feature_encoder."))
print("feature_encoder params (should be ~1e4, non-zero):", fe_params)

@torch.no_grad()
def sjepa_embeddings(X):
    # X: trials x C x T  ->  trials x (C*d)  (mean-pooled over time tokens, per channel)
    xb=torch.as_tensor(np.asarray(X,np.float32),device=DEVICE)
    feat=ENCODER.feature_encoder(xb)              # [B, C*t, d]  (channel-major)
    B=feat.shape[0]; d=feat.shape[-1]; L=feat.shape[1]; t=L//N_CH
    feat=feat.reshape(B,N_CH,t,d).mean(dim=2)     # [B, C, d]
    return feat.reshape(B,-1).cpu().numpy().astype(np.float64)
# probe
_e=sjepa_embeddings(SUBJECTS[ALL[0]][0][:2]); print("embedding dim:",_e.shape[1])


# 5. Riemannian branch: filter-bank tangent-space features

In [ ]:
def bandpass(X, lo, hi):
    return mne.filter.filter_data(X, SFREQ, lo, hi, method="fir", phase="zero", fir_design="firwin", verbose=False)

def fit_riemann(X_train):
    # returns list of (band, fitted Covariances-on-the-fly via transform, fitted TangentSpace)
    fitted=[]
    for (lo,hi) in CONFIG["filter_bands"]:
        Xb=bandpass(X_train, lo, hi)
        cov=Covariances(estimator=CONFIG["cov_estimator"]).transform(Xb)
        ts=TangentSpace().fit(cov)
        fitted.append(((lo,hi), ts))
    return fitted

def transform_riemann(X, fitted):
    feats=[]
    for (lo,hi),ts in fitted:
        Xb=bandpass(X, lo, hi)
        cov=Covariances(estimator=CONFIG["cov_estimator"]).transform(Xb)
        feats.append(ts.transform(cov))
    return np.concatenate(feats, axis=1)


# 6. Evaluate the three pipelines on the same folds

In [ ]:
def lda(): return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=CONFIG["lda_shrinkage"])

def eval_subject(sid):
    X,y=SUBJECTS[sid]
    skf=StratifiedKFold(n_splits=CONFIG["cv_splits"], shuffle=True, random_state=CONFIG["seed"])
    preds={"riemannian":[], "sjepa":[], "fusion":[]}; truth=[]
    for tr,te in skf.split(X,y):
        truth.extend(y[te].tolist())
        # --- riemannian features ---
        rf=fit_riemann(X[tr]); Rtr=transform_riemann(X[tr],rf); Rte=transform_riemann(X[te],rf)
        srt=StandardScaler().fit(Rtr); Rtr_s,Rte_s=srt.transform(Rtr),srt.transform(Rte)
        # --- sjepa features ---
        Etr=sjepa_embeddings(X[tr]); Ete=sjepa_embeddings(X[te])
        set_=StandardScaler().fit(Etr); Etr_s,Ete_s=set_.transform(Etr),set_.transform(Ete)
        # --- classifiers ---
        preds["riemannian"].extend(lda().fit(Rtr_s,y[tr]).predict(Rte_s).tolist())
        preds["sjepa"].extend(lda().fit(Etr_s,y[tr]).predict(Ete_s).tolist())
        Ftr=np.concatenate([Rtr_s,Etr_s],1); Fte=np.concatenate([Rte_s,Ete_s],1)
        preds["fusion"].extend(lda().fit(Ftr,y[tr]).predict(Fte).tolist())
    yt=np.array(truth)
    out={"subject":int(sid)}
    for k in preds:
        yp=np.array(preds[k])
        out[k]=float(balanced_accuracy_score(yt,yp))
        out[f"{k}_pred"]=yp.tolist()
    out["y_true"]=yt.tolist()
    return out

RESULTS=[]
for i,s in enumerate(ALL,1):
    r=eval_subject(s); RESULTS.append(r)
    print(f"[{i:2d}/{len(ALL)}] subj {s:2d}  riem={r['riemannian']*100:5.1f}  "
          f"sjepa={r['sjepa']*100:5.1f}  fusion={r['fusion']*100:5.1f}")
with open(ART/"hybrid_results.json","w") as f: json.dump(RESULTS,f,indent=2)


# 7. Side-by-side comparison

In [ ]:
def global_ba(key):
    yt=np.concatenate([r["y_true"] for r in RESULTS]); yp=np.concatenate([r[f"{key}_pred"] for r in RESULTS])
    return balanced_accuracy_score(yt,yp)
rows=[]
for key in ["riemannian","sjepa","fusion"]:
    arr=np.array([r[key] for r in RESULTS])
    rows.append({"pipeline":key,"mean_per_subject_BA_%":round(arr.mean()*100,2),
                 "SD_%":round(arr.std()*100,2),"global_pooled_BA_%":round(global_ba(key)*100,2)})
summary=pd.DataFrame(rows)
print("="*64)
print(f"S-JEPA x Riemannian hybrid | {len(ALL)} subjects | {CONFIG['cv_splits']}-fold within-subject | "
      f"pretrained={CONFIG['pretrained_mode']}")
print(summary.to_string(index=False))
print("Reference: Liu CSP+LDA 55.57 | FBCSP+SVM 57.57 | TWFB+DGFMDM 72.21")
print("="*64)
summary.to_csv(ART/"hybrid_summary.csv",index=False)
per_subj=pd.DataFrame([{"subject":r["subject"],"riemannian_%":round(r["riemannian"]*100,1),
    "sjepa_%":round(r["sjepa"]*100,1),"fusion_%":round(r["fusion"]*100,1)} for r in RESULTS]).sort_values("subject")
per_subj.to_csv(ART/"hybrid_per_subject.csv",index=False)
summary


## How to read this
- **fusion vs riemannian**: does adding S-JEPA features help, hurt, or do nothing on top of the Riemannian branch?
- **sjepa vs riemannian**: how far the pretrained features get on their own with a data-efficient head (and
  whether that beats your collapse-prone SGD-head runs at ~57%).
- **pretrained vs random**: set `pretrained_mode="random"` to confirm the S-JEPA contribution is real.
- For the strongest standalone Riemannian number (per-subject TW+FB selection + FgMDM), compare against
  `liu2024_twfb_dgfmdm_reproduction.ipynb`, which is the closest reproduction of the 72% method.